# Fraud Detection Modeling

In [1]:
# Strongest features shoud be fraud_risk, previous_fraud_flag, is_international, unusual_location
# Then, multiple_transactions_short_time, unusual_amount_flag, velocity_flag, and transaction_frequency_24h

## Import libraries

In [14]:
import pandas as pd
import numpy as np

## Data loading

In [3]:
df = pd.read_csv('../data/retail_fraud_detection_100k.csv')

In [4]:
df.head()

,transaction_id,customer_id,transaction_timestamp,transaction_amount,payment_method,device_type,location,merchant_category,is_international,transaction_frequency_24h,...,failed_transaction_count_24h,account_age_days,previous_fraud_flag,unusual_amount_flag,unusual_location_flag,multiple_transactions_short_time,high_risk_device_flag,velocity_flag,fraud_flag,fraud_risk
0,T0000001,C20953,2026-03-04 19:56:23.679173,56.31,Debit Card,Tablet,Canada,Fashion,0,13,...,4,1054,0,0,0,1,0,1,0,Medium
1,T0000002,C24133,2026-03-09 19:56:23.679330,20.35,Credit Card,Tablet,India,Electronics,0,3,...,2,97,0,0,0,0,0,0,0,Low
2,T0000003,C07165,2026-01-01 19:56:23.679362,48.72,Debit Card,Tablet,UK,Travel,0,8,...,4,779,1,0,0,0,0,0,0,Medium
3,T0000004,C19310,2025-12-09 19:56:23.679385,153.62,Debit Card,Mobile,Australia,Fashion,0,14,...,3,286,1,1,0,1,0,1,1,High
4,T0000005,C25019,2025-11-09 19:56:23.679409,115.32,PayPal,Mobile,India,Gaming,0,10,...,3,866,0,0,0,1,0,0,0,Low


In [5]:
df.dtypes

transaction_id                       object
customer_id                          object
transaction_timestamp                object
transaction_amount                  float64
payment_method                       object
device_type                          object
location                             object
merchant_category                    object
is_international                      int64
transaction_frequency_24h             int64
avg_transaction_amount_7d           float64
failed_transaction_count_24h          int64
account_age_days                      int64
previous_fraud_flag                   int64
unusual_amount_flag                   int64
unusual_location_flag                 int64
multiple_transactions_short_time      int64
high_risk_device_flag                 int64
velocity_flag                         int64
fraud_flag                            int64
fraud_risk                           object
dtype: object

In [9]:
cols_change = ['previous_fraud_flag', 'is_international', 'unusual_location_flag', 'multiple_transactions_short_time', 'unusual_amount_flag', 'velocity_flag']

df[cols_change] = df[cols_change].astype(object)

## Model training

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

X = df.drop('fraud_flag', axis=1)
y = df['fraud_flag']

cat_cols = ['fraud_risk', 'previous_fraud_flag', 'is_international', 'unusual_location_flag', 'multiple_transactions_short_time', 'unusual_amount_flag', 'velocity_flag']
num_cols = ['transaction_frequency_24h']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Linear Regression

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

lm = Pipeline([
    ('prep', preprocessor),
    ('lin', LinearRegression())
])

lm.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['transaction_frequency_24h']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['fraud_risk',
                                                   'previous_fraud_flag',
                                                   'is_international',
                                                   'unusual_location_flag',
                                                   'multiple_transactions_short_time',
                                                   'unusual_amount_flag',
                                                   'velocity_flag'])])),
                ('lin', LinearRegression())])

In [16]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, X_train, X_text, y_train, y_test, X_full=None, y_full=None):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    if X_full is not None and y_full is not None:
        y_pred_full = model.predict(X_full)
        print('Model score (full data) R^2: \t{:.4f}'.format(model.score(X_full, y_full)))
        
    print('R^2 score on training data: \t{:.4f}'.format(model.score(X_train, y_train)))
    print('R^2 score on test data: \t{:.4f}'.format(model.score(X_test, y_test)))
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)
    
    print('RMSE on test data: \t{:.4f}'.format(rmse))
    print('Mean Absolute Error: \t{:.4f}'.format(mae))
    
evaluate_model(lm, X_train, X_test, y_train, y_test, X, y)

Model score (full data) R^2: 	0.6202
R^2 score on training data: 	0.6187
R^2 score on test data: 	0.6264
RMSE on test data: 	0.3054
Mean Absolute Error: 	0.2519


### Random Forest

In [17]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline([
    ('prep', preprocessor),
    ('ran_for', RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['transaction_frequency_24h']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['fraud_risk',
                                                   'previous_fraud_flag',
                                                   'is_international',
                                                   'unusual_location_flag',
                                                   'multiple_transactions_short_time',
                                                   'unusual_amount_flag',
                                                   'velocity_flag'])])),
                ('ran_for',
                 RandomForestRegressor(max_depth=15, min_samples_leaf=5,
                                       n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [19]:
evaluate_model(rf, X_train, X_test, y_train, y_test, X, y)

Model score (full data) R^2: 	0.7297
R^2 score on training data: 	0.7278
R^2 score on test data: 	0.7372
RMSE on test data: 	0.2562
Mean Absolute Error: 	0.1327


### XGBoost

In [20]:
import xgboost as xgb

xgb_model = Pipeline([
    ('prep', preprocessor),
    ('xgb_m', xgb.XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.03,
        random_state=42,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['transaction_frequency_24h']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['fraud_risk',
                                                   'previous_fraud_flag',
                                                   'is_international',
                                                   'unusual_location_flag',
                                                   'multiple_transactions_short_time',
                                                   'unusual_amount_flag',
                                                   'velocity_flag'])])),
                ('xgb_m',
                 XGBRegressor(b...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [21]:
evaluate_model(xgb_model, X_train, X_test, y_train, y_test, X, y)

Model score (full data) R^2: 	0.7297
R^2 score on training data: 	0.7278
R^2 score on test data: 	0.7373
RMSE on test data: 	0.2561
Mean Absolute Error: 	0.1340
